# DNN — model selection

Forecasts are saved under `results/dnn/<split>/<run>/<model>.csv`:

- `val/screening` — one week from each validation month
- `val/full` — every day of the validation set
- `test/full` — backtest on the test set

`calibration_window` is in years of 364 days, so `2` is the same history as LEAR's 728 days. Hyperparameters are tuned on the training set only. The trials file for each config is `data/dnn/hyperparameters/`.

DNN is the fully connected network from `epftoolbox` with two hidden layers, trained with the MAE loss, and it forecasts all 24 hours of the next day at once. It can use the same inputs as LEAR (see `01_LEAR.ipynb`), and which of them it uses is part of the hyperparameter search. Once the search is done the hyperparameters stay fixed. Every validation day the network is retrained with them on the last `calibration_window` years, and a random quarter of the training weeks is held out for early stopping.

The forecasts need history before every validation day, so they are made on `dataset_full`, the train and validation sets joined, while tuning uses `dataset_train` only.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

import pandas as pd

from src.config import EPF_EXOG
from src.utils.epf import (
    evaluate_dnn,
    forecast_path,
    tune_dnn,
)
from src.utils.evaluation import compare_models, load_forecasts
from src.utils.file_io import load_final_dataset

/opt/homebrew/Caskroom/miniforge/base/envs/epf-lear-dnn/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
dataset_train = load_final_dataset("epf", "pl", "train")
dataset_val = load_final_dataset("epf", "pl", "val")
dataset_full = pd.concat([dataset_train, dataset_val])
prices = dataset_full["PL_price"]

configs = [
    # {"model": "DNN_1", "calibration_window": 1, "exog": EPF_EXOG},
    # {"model": "DNN_2", "calibration_window": 2, "exog": EPF_EXOG},
    # {"model": "DNN_3", "calibration_window": 3, "exog": EPF_EXOG},
    {"model": "DNN_3_n", "calibration_window": 3, "exog": ["gen_PL", ""]},
]

val_start = dataset_val.index.min()
val_end = dataset_val.index.max()
month_starts = list(pd.date_range(val_start, val_end, freq="MS"))

max_evals = 300

## 1. Hyperparameter tuning

One search per config, on `dataset_train` only.

The search (hyperopt, TPE) chooses the network hyperparameters and the input features, with up to `max_evals` trials per config. A trial is scored by the MAE on training weeks held out for validation. The trials are saved to a file, and `new_hyperopt=0` continues the search from that file instead of starting a new one.

In [ ]:
for config in configs:
    tune_dnn(
        dataset=dataset_train,
        exog=config["exog"],
        calibration_window=config["calibration_window"],
        experiment_id=config["model"],
        max_evals=max_evals,
        new_hyperopt=0,
    )

## 2. Full validation

The tuned hyperparameters stay fixed. For each of the 126 validation days the network is retrained on the last `calibration_window` years and forecasts the next 24 hours. A day that fails with an error during training is retried. The forecasts are saved to `results/dnn/val/full/<model>.csv`.

In [ ]:
for config in configs:
    evaluate_dnn(
        dataset=dataset_full,
        calibration_window=config["calibration_window"],
        exog=config["exog"],
        forecast_start=val_start,
        forecast_end=val_end,
        experiment_id=config["model"],
        path=forecast_path(config["model"], split="val", run="full", family="dnn"),
    )

## 3. Full validation comparison

Same metrics as for LEAR: MAE and RMSE in EUR/MWh, and rMAE relative to the weekly naive forecast. Lower is better for all three.

In [3]:
full_models = [config["model"] for config in configs]
forecasts = load_forecasts(
    [
        forecast_path(model, split="val", run="full", family="dnn")
        for model in full_models
    ]
)

compare_models(forecasts, prices, metrics=["rMAE", "MAE", "RMSE"])

,rMAE,MAE,RMSE
model,,,
DNN_1,0.698246,18.902479,30.957843
DNN_2,0.686614,18.587599,29.724201
DNN_3,0.685365,18.553771,30.829777


## 4. Conclusions

One year of history is the weakest (`DNN_1`, rMAE 0.698). `DNN_2` and `DNN_3` are close: `DNN_3` has the lowest rMAE and MAE (0.685 and 18.55), but `DNN_2` has a lower RMSE (29.7 against 30.8), so the third year does not give a clear gain over two.

The DNN is compared with LEAR in `03_LEAR_DNN_comparison.ipynb`.